# Chapter 12 &mdash; More Context Is Not a Stack

**Concept 13 of the Chapter 12 decomposition:** *More Context Is Not a Stack*

Widen the window from 3 to 8 and the loss improves, the samples improve &mdash; and then the strings get longer and it all comes apart.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Concept-Notebooks/Chapter12-PDA/Concept-More-Context-Is-Not-A-Stack/Concept-More-Context-Is-Not-A-Stack.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.Def_PDA        import *
from jove.LangDef        import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


The obvious objection to the previous notebook is that three symbols is a mean
allowance. Give the model a longer window and surely it can do brackets.

It can do *better*. That is not the same thing, and this notebook is about the
difference.

A window of $k$ symbols lets the model keep a bracket count for nesting up to about
$k$ &mdash; because when the whole relevant history fits inside the window, the model
can simply read it. That is a **bounded** counter. A bounded counter is a DFA, which
is exactly the thing Chapter 6's model already was.

So the prediction is:

* widen the window, and the model gets better **at the lengths it was trained on**;
* run it past the window, and the improvement evaporates.

Both halves get measured, and the notebook closes by putting the same question to
the PDA. `STKMAX` looks like Jove's own version of a context window, so it is worth
checking whether bounding it breaks the PDA the way bounding $k$ breaks the model.
It does not, and Concept 7 already said why.

## 2. Definitions

### The model, the training loop, the sampler

In [ ]:
#@title minimal GPT implementation in PyTorch  (Andrej Karpathy)
#
# Read it, change it, break it.  This is the whole model: an embedding, a
# few attention blocks, a linear head.  Nothing here knows about automata.
""" super minimal decoder-only gpt """

import math
from dataclasses import dataclass
import torch
import torch.nn as nn
from torch.nn import functional as F

class CausalSelfAttention(nn.Module):

    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        # key, query, value projections for all heads, but in a batch
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        # output projection
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        # regularization
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                                    .view(1, 1, config.block_size, config.block_size))

    def forward(self, x):
        B, T, C = x.size() # batch size, sequence length, embedding dimensionality (n_embd)

        # calculate query, key, values for all heads in batch and move head forward to be the batch dim
        q, k ,v  = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)

        # manual implementation of attention
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
        att = att.masked_fill(self.bias[:,:,:T,:T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        y = att @ v # (B, nh, T, T) x (B, nh, T, hs) -> (B, nh, T, hs)
        y = y.transpose(1, 2).contiguous().view(B, T, C) # re-assemble all head outputs side by side

        # output projection
        y = self.c_proj(y)
        return y

class MLP(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.c_fc    = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.c_proj  = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        self.nonlin = nn.GELU()

    def forward(self, x):
        x = self.c_fc(x)
        x = self.nonlin(x)
        x = self.c_proj(x)
        return x

class Block(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

@dataclass
class GPTConfig:
    # these are default GPT-2 hyperparameters
    block_size: int = 1024
    vocab_size: int = 50304
    n_layer: int = 12
    n_head: int = 12
    n_embd: int = 768
    bias: bool = False

class GPT(nn.Module):

    def __init__(self, config):
        super().__init__()
        assert config.vocab_size is not None
        assert config.block_size is not None
        self.config = config

        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.n_embd),
            wpe = nn.Embedding(config.block_size, config.n_embd),
            h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f = nn.LayerNorm(config.n_embd),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight # https://paperswithcode.com/method/weight-tying

        # init all weights
        self.apply(self._init_weights)
        # apply special scaled init to the residual projections, per GPT-2 paper
        for pn, p in self.named_parameters():
            if pn.endswith('c_proj.weight'):
                torch.nn.init.normal_(p, mean=0.0, std=0.02/math.sqrt(2 * config.n_layer))

        # report number of parameters
        print("number of parameters: %d" % (sum(p.nelement() for p in self.parameters()),))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx):
        device = idx.device
        b, t = idx.size()
        assert t <= self.config.block_size, f"Cannot forward sequence of length {t}, block size is only {self.config.block_size}"
        pos = torch.arange(0, t, dtype=torch.long, device=device).unsqueeze(0) # shape (1, t)

        # forward the GPT model itself
        tok_emb = self.transformer.wte(idx) # token embeddings of shape (b, t, n_embd)
        pos_emb = self.transformer.wpe(pos) # position embeddings of shape (1, t, n_embd)
        x = tok_emb + pos_emb
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)
        logits = self.lm_head(x[:, -1, :]) # note: only returning logits at the last time step (-1), output is 2D (b, vocab_size)
        return logits

&nbsp;

In [ ]:
def make_XY(seq, context_length):
    X, Y = [], []
    for i in range(len(seq) - context_length):
        X.append(seq[i:i + context_length])
        Y.append(seq[i + context_length])
    return (torch.tensor(X, dtype=torch.long),
            torch.tensor(Y, dtype=torch.long))

def train_gpt(gpt, X, Y, iters=200, lr=1e-3, every=20):
    optimizer = torch.optim.AdamW(gpt.parameters(), lr=lr, weight_decay=1e-1)
    losses = []
    for i in range(iters):
        logits = gpt(X)
        loss = F.cross_entropy(logits, Y)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        losses.append(loss.item())
        if i % every == 0 or i == iters - 1:
            print(i, loss.item())
    return losses

&nbsp;

In [ ]:
# --- sample from the model, exactly as Karpathy does --------------------
def sample(gpt, start, steps=24):
    xi = list(start)
    full = xi.copy()
    for _ in range(steps):
        x = torch.tensor(xi, dtype=torch.long)[None, ...]
        probs = nn.functional.softmax(gpt(x), dim=-1)
        t = torch.multinomial(probs[0], num_samples=1).item()
        xi = xi[1:] + [t]
        full.append(t)
    return ''.join(map(str, full))

### The corpus, from a PDA

In [ ]:
# --- the training data: strings the PDA accepts, run together ------------
from functools import reduce

def pda_accepts(P, s, STKMAX=20):
    surv, paths, visited = run_pda(s, P, acceptance='ACCEPT_F',
                                   STKMAX=STKMAX, chatty=False)
    return len(paths) > 0        # accepted iff SOME path ended in F

def corpus_from(P, upto=16000, STKMAX=8):
    strings = [nthnumeric(i, ['0', '1']) for i in range(upto)]
    good = [s for s in strings if pda_accepts(P, s, STKMAX)]
    return good, list(map(int, reduce(lambda a, b: a + b, good)))

### The language, and the measurements

In [ ]:
# --- the language: balanced brackets, with 0 for '(' and 1 for ')' -------
# The stack is doing the one thing no DFA can do: counting with no bound.
DYCK = md2mc('''PDA
IF : 0 , # ; 0# -> A
A  : 0 , 0 ; 00 -> A
A  : 1 , 0 ; '' -> A
A  : '' , # ; # -> IF
''')

# --- how much probability does the model give a FORBIDDEN symbol? --------
# Where the stack is empty the language cannot close, so '1' is forbidden.
# Where a capped stack is full it cannot open, so '0' is forbidden.  Ask
# the model how much mass it puts there.  No threshold and no score: one
# number, between 0 and 1, saying how much of the rule it has picked up.
def forbidden_mass(gpt, k, good, cap=None):
    total, n = 0.0, 0
    for s in good:
        depth = 0
        for i, c in enumerate(s):
            bad = 1 if depth == 0 else (0 if depth == cap else None)
            if bad is not None and i >= k:
                x = torch.tensor([int(ch) for ch in s[i - k:i]],
                                 dtype=torch.long)[None, ...]
                p = nn.functional.softmax(gpt(x), dim=-1)[0].tolist()
                total += p[bad]
                n += 1
            depth += 1 if c == '0' else -1
    return total / max(n, 1), n

# --- generate a string, then let the PDA be the judge --------------------
# The seed is the same SHAPE for every context length -- a shallow prefix
# 0101... -- so that changing k does not quietly change the question.
def seed_for(k):
    return [int(ch) for ch in ('01' * k)[:k]]

def try_samples(gpt, k, P, length=16, n=30, seed=0):
    torch.manual_seed(seed)
    got = [sample(gpt, seed_for(k), steps=length - k) for _ in range(n)]
    ok = [s for s in got if pda_accepts(P, s)]
    return got, ok

<!-- nav-strip -->

---

&larr;&nbsp;[Ch12&nbsp;12.&nbsp;A Transformer on a Language That Needs a Stack](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Concept-Notebooks/Chapter12-PDA/Concept-Karpathy-GPT-On-A-Jove-PDA/Concept-Karpathy-GPT-On-A-Jove-PDA.ipynb) &nbsp;&middot;&nbsp; [**Chapter 12** index](https://github.com/ganeshutah/Jove/blob/master/Chapter12-PDA/README.md) &nbsp;&middot;&nbsp; [Ch12&nbsp;14.&nbsp;Bound the Stack and the Model Learns It](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Concept-Notebooks/Chapter12-PDA/Concept-Bounding-The-Stack/Concept-Bounding-The-Stack.ipynb)&nbsp;&rarr;

---

## 3. Tests

One corpus, shared by every model below.

In [ ]:
good, seq = corpus_from(DYCK)
print('%d strings, longest %d, %d training symbols'
      % (len(good), max(map(len, good)), len(seq)))

def train_at(k, iters=200, n_embd=16, seed=1337):
    X, Y = make_XY(seq, k)
    config = GPTConfig(block_size=k, vocab_size=2, n_layer=4, n_head=4,
                       n_embd=n_embd, bias=False)
    torch.manual_seed(seed)
    gpt = GPT(config)
    losses = train_gpt(gpt, X, Y, iters=iters, every=iters)
    return gpt, losses[-1]

**Three windows.** Loss, and the probability mass still going to a symbol the language forbids.

In [ ]:
models = {}
print('%-6s %-10s %s' % ('window', 'loss', 'mass on a FORBIDDEN symbol'))
for k in (3, 5, 8):
    gpt, fl = train_at(k)
    mass, spots = forbidden_mass(gpt, k, good)
    models[k] = gpt
    print('k=%-4d %-10.4f %.3f   (over %d places)' % (k, fl, mass, spots))

Both numbers improve. A wider window really is buying something &mdash; so the interesting question is *what*.

In [ ]:
print("A coin sits at 0.500.  Every model above is well under that, and the")
print("bigger the window the further under it gets.")
print()
print("Now ask WHERE the improvement came from.")

**The test that tells them apart.** Generate strings of increasing length and let the PDA mark them. Everything in the corpus was length 12 or shorter, so 24 and 32 are new territory.

In [ ]:
print('%-8s %s' % ('', '   '.join('L=%-2d' % L for L in (12, 16, 24, 32))))
for k in (3, 5, 8):
    row = []
    for L in (12, 16, 24, 32):
        got, ok = try_samples(models[k], k, DYCK, length=L, n=30)
        row.append('%2d/30' % len(ok))
    print('k=%-6d %s' % (k, '   '.join(row)))

Read the rows left to right, then read the columns top to bottom.

In [ ]:
print("Left to right, every row falls away.  The widest window holds up")
print("longest -- and then it falls away too.")
print()
print("At length 12 with k=8 the model can see two thirds of the string it")
print("is completing.  It is not remembering how many brackets are open; it")
print("is LOOKING at them.  Push the length out and there is less and less")
print("to look at, and the advantage goes with it.")
print()
print("That is the difference between a window and a stack.  A window of 8")
print("can imitate a stack of depth 8.  A stack has no number in it at all.")

Now the same question asked of the PDA, which is the control this notebook needs. `STKMAX` looks like the PDA's version of a context window &mdash; so does bounding it break the PDA the same way?

In [ ]:
print('%-10s %s' % ('', '  '.join('STKMAX=%-2d' % k for k in (2, 4, 6, 20))))
for n in (6, 12, 20):
    deep = '0' * n + '1' * n
    print('%-10s %s' % ('depth %d' % n,
          '  '.join('%-9s' % pda_accepts(DYCK, deep, STKMAX=k)
                    for k in (2, 4, 6, 20))))

It does not, and that is the whole contrast.

In [ ]:
print("Nesting 20 deep is accepted with STKMAX=2.  Concept 7 said this in")
print("so many words: STKMAX is NOT an absolute depth cap.  It prunes an ID")
print("whose stack has grown at a configuration already seen -- it stops")
print("epsilon-loops from pushing forever.  A stack doing honest work is")
print("never truncated, however deep it goes.")
print()
print("So the PDA has no number in it that the language can outgrow, and")
print("the transformer has k.  That is not a difference of degree.")
print()
print("Which is why the previous notebook's forbidden-symbol mass stalls")
print("and this one's acceptance rate decays: not because 8 is too small a")
print("window, but because there is no window that is large enough.")

## 4. Exercises


1. Add `k=12` to the table. The corpus strings are at most 12 long, so the window now
   covers a whole string. Predict the `L=12` column before you run it, then the
   `L=32` column.
2. Raise `n_embd` from 16 to 64 at `k=3`. Does width substitute for window? Why would
   you expect not?
3. The `L=12` column is the only one drawn from the training distribution. Which
   column would you quote if you wanted the model to look good, and which if you
   wanted the truth?
4. Build a corpus of balanced strings up to length 20 by raising `upto`. How long does
   `corpus_from` take, and what does that say about testing on longer strings still?
5. `STKMAX` and the context window are both "a number that bounds the machine", yet
   only one of them limits the language recognised. Say precisely what `STKMAX`
   bounds, and why that is not the stack's depth.

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for every concept.
# Type a chapter (Chapter7, ch7, NFA) or words from a title (pumping).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:
#     load_here('Chapter7-NFA/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter12-PDA/Concept-More-Context-Is-Not-A-Stack')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')